In [9]:
# pip install geopandas pandas fiona shapely pyproj
import geopandas as gpd
import pandas as pd
import fiona
from shapely.geometry import MultiPolygon, mapping
import os

repopath = r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Teste_AutomatizaçãoShapesCORSAN_cBruninha\Shapes bacias R00 - teste"

CAMINHO_BACIAS         = os.path.join(repopath, "Bacias.gpkg")
CAMINHO_EBACIA_SEMENTE = os.path.join(repopath, "E_BACIA_Semente.gpkg")
CAMINHO_PLANILHA       = os.path.join(repopath, "Bacias.xlsx")
SAIDA_GPKG             = os.path.join(repopath, "E_BACIA.gpkg")

# === 1. Ler os dados ===
gdf_bacias = gpd.read_file(CAMINHO_BACIAS)
df_xls = pd.read_excel(CAMINHO_PLANILHA)

# Padronizar texto na coluna de junção
gdf_bacias["NOME"] = gdf_bacias["NOME"].astype(str).str.strip()
df_xls["NOME"] = df_xls["NOME"].astype(str).str.strip()

# === 2. Fazer o join (atributos + geometria) ===
gdf_join = gdf_bacias.merge(df_xls, on="NOME", how="left")

# === 3. Ler o schema do E_BACIA_Semente ===
with fiona.open(CAMINHO_EBACIA_SEMENTE) as src:
    schema = src.schema
    crs_wkt = src.crs_wkt
    geom_exp = schema["geometry"]
    props = schema["properties"]

# === 4. Ajustar geometrias se necessário ===
def ajustar_geom(g):
    if g is None:
        return None
    if geom_exp.lower() == "multipolygon" and g.geom_type == "Polygon":
        return MultiPolygon([g])
    return g

gdf_join["geometry"] = gdf_join.geometry.apply(ajustar_geom)

# === 5. Manter apenas as colunas do E_BACIA ===
cols_ebacia = list(props.keys())
df_attrs = gdf_join.reindex(columns=cols_ebacia + ["geometry"])

# === 6. Criar GeoDataFrame final ===
gdf_out = gpd.GeoDataFrame(df_attrs, geometry="geometry", crs=gdf_bacias.crs)

# === 7. Escrever para o novo GPKG preservando o schema ===
with fiona.open(
    SAIDA_GPKG, mode="w", driver="GPKG", schema=schema, crs_wkt=crs_wkt
) as dst:
    for _, row in gdf_out.iterrows():
        props_row = {k: (None if pd.isna(v) else v) for k, v in row[cols_ebacia].to_dict().items()}
        geom_row = None if row.geometry is None else mapping(row.geometry)
        dst.write({"geometry": geom_row, "properties": props_row})

print("✅ E_BACIA.gpkg criado com sucesso!")


✅ E_BACIA.gpkg criado com sucesso!


In [10]:
# pip install geopandas pandas fiona shapely pyproj
import os
import pandas as pd
import geopandas as gpd
import fiona
from shapely.geometry import (
    Point, MultiPoint, LineString, MultiLineString, Polygon, MultiPolygon
)
from shapely.geometry import mapping
# from shapely.ops import unary_union  # use se quiser mesclar linhas/polígonos

def _ajustar_geom_para_tipo(g, geom_exp: str):
    """Converte a geometria 'g' para o tipo 'geom_exp' (do schema do semente) quando possível."""
    if g is None:
        return None
    exp = (geom_exp or "").lower()
    gtype = (g.geom_type or "").lower()

    # --- Pontos ---
    if exp == "point":
        if gtype == "point":
            return g
        if gtype == "multipoint":
            # escolha simples: pegar o primeiro ponto
            geoms = list(g.geoms)
            return geoms[0] if geoms else None
        return g  # fallback

    if exp == "multipoint":
        if gtype == "point":
            return MultiPoint([g])
        if gtype == "multipoint":
            return g
        return g  # fallback

    # --- Linhas ---
    if exp == "linestring":
        if gtype == "linestring":
            return g
        if gtype == "multilinestring":
            # opção simples: pegar a primeira linha (ou usar unary_union para fundir)
            geoms = list(g.geoms)
            return geoms[0] if geoms else None
        return g

    if exp == "multilinestring":
        if gtype == "linestring":
            return MultiLineString([g])
        if gtype == "multilinestring":
            return g
        return g

    # --- Polígonos ---
    if exp == "polygon":
        if gtype == "polygon":
            return g
        if gtype == "multipolygon":
            # escolha simples: maior área
            geoms = list(g.geoms)
            return max(geoms, key=lambda p: p.area) if geoms else None
        return g

    if exp == "multipolygon":
        if gtype == "polygon":
            return MultiPolygon([g])
        if gtype == "multipolygon":
            return g
        return g

    # Desconhecido / já compatível
    return g


def preencher_de_semente(
    caminho_gpkg_entrada: str,
    layer_entrada: str,
    caminho_gpkg_semente: str,
    layer_semente: str,
    caminho_planilha: str,
    sheet_planilha=0,
    chave_gis="NOME",
    chave_xls="NOME",
    caminho_saida_gpkg: str = None,
    layer_saida: str = None,
    validar_one_to_one=False,
    forcar_crs_do_semente_na_saida=True,
):
    """
    Preenche uma nova camada GPKG a partir de:
      - geometria + chave: layer de entrada (gpkg)
      - schema/tipo de geometria/CRS: layer semente (gpkg)
      - atributos: planilha (mesmos nomes de campos do semente)
    Regras:
      - join pela chave (padronizada como string .strip())
      - mantém APENAS os campos do semente (na mesma ordem)
      - ajusta tipo geométrico conforme schema do semente
      - grava com o schema e CRS do semente
    """
    if caminho_saida_gpkg is None:
        # cria um nome padrão ao lado do semente
        base_dir = os.path.dirname(os.path.abspath(caminho_gpkg_semente))
        layer_nm = layer_saida or layer_semente or "saida"
        caminho_saida_gpkg = os.path.join(base_dir, f"{layer_nm}.gpkg")

    # 1) Ler entrada e planilha
    gdf_in = gpd.read_file(caminho_gpkg_entrada, layer=layer_entrada)
    df_xls = pd.read_excel(caminho_planilha, sheet_name=sheet_planilha)

    # 2) Padronizar chaves
    gdf_in[chave_gis] = gdf_in[chave_gis].astype(str).str.strip()
    df_xls[chave_xls] = df_xls[chave_xls].astype(str).str.strip()

    # 3) Merge (join) - opcional validação de unicidade
    validate = "one_to_one" if validar_one_to_one else None
    gdf_join = gdf_in.merge(df_xls, left_on=chave_gis, right_on=chave_xls, how="left", validate=validate)

    # 4) Ler schema e CRS do semente
    with fiona.open(caminho_gpkg_semente, layer=layer_semente) as src:
        schema = src.schema
        crs_wkt = src.crs_wkt
        geom_exp = schema["geometry"]
        props = schema["properties"]

    # 5) Ajustar geometria para o tipo esperado
    gdf_join["geometry"] = gdf_join.geometry.apply(lambda g: _ajustar_geom_para_tipo(g, geom_exp))

    # 6) Manter apenas os campos do semente, na ordem
    cols_modelo = list(props.keys())
    df_attrs = gdf_join.reindex(columns=cols_modelo + ["geometry"])

    # (Opcional) coerção leve de datas: se o schema tem 'date'/'datetime'
    for campo, tipo in props.items():
        t = (tipo or "").lower()
        if campo in df_attrs.columns and t in ("date", "datetime"):
            df_attrs[campo] = pd.to_datetime(df_attrs[campo], errors="coerce")
            if t == "date":
                # grava apenas a data (sem hora)
                df_attrs[campo] = df_attrs[campo].dt.date

    # 7) Construir GeoDataFrame final
    gdf_out = gpd.GeoDataFrame(df_attrs, geometry="geometry", crs=gdf_in.crs)

    # (Opcional) Reprojetar para o CRS do semente, se quiser coordenadas 100% no mesmo sistema
    if forcar_crs_do_semente_na_saida and crs_wkt:
        try:
            gdf_out = gdf_out.to_crs(crs_wkt)
        except Exception:
            # Se falhar (ex.: wkt não interpretável pelo pyproj), segue sem reprojetar
            pass

    # 8) Escrever preservando exatamente o schema do semente
    #    Use 'layer=' para controlar o nome da layer no GPKG de saída
    with fiona.open(
        caminho_saida_gpkg,
        mode="w",
        driver="GPKG",
        schema=schema,
        crs_wkt=crs_wkt,
        layer=(layer_saida or layer_semente),
    ) as dst:
        for _, row in gdf_out.iterrows():
            props_row = {k: (None if pd.isna(v) else v) for k, v in row[cols_modelo].to_dict().items()}
            geom_row = None if row.geometry is None else mapping(row.geometry)
            dst.write({"geometry": geom_row, "properties": props_row})

    print(f"✅ Gravado: {caminho_saida_gpkg} (layer='{layer_saida or layer_semente}')")
    return caminho_saida_gpkg


In [11]:
repopath = r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Teste_AutomatizaçãoShapesCORSAN_cBruninha\Shapes bacias R00 - teste"

In [16]:
# 1) BACIAS (polígono)
preencher_de_semente(
    caminho_gpkg_entrada = os.path.join(repopath, "Bacias.gpkg"),
    layer_entrada        = "Bacias",              # ajuste se o nome da layer for outro
    caminho_gpkg_semente = os.path.join(repopath, "E_BACIA_Semente.gpkg"),
    layer_semente        = "E_BACIA_Semente",             # ajuste ao nome real da layer do semente
    caminho_planilha     = os.path.join(repopath, "Bacias.xlsx"),
    sheet_planilha       = 0,
    chave_gis            = "NOME",
    chave_xls            = "NOME",
    caminho_saida_gpkg   = os.path.join(repopath, "E_BACIA.gpkg"),
    layer_saida          = "E_BACIA",
)

✅ Gravado: C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Teste_AutomatizaçãoShapesCORSAN_cBruninha\Shapes bacias R00 - teste\E_BACIA.gpkg (layer='E_BACIA')


'C:\\Users\\gabriel.coimbra\\Desktop\\Meus arquivos\\Teste_AutomatizaçãoShapesCORSAN_cBruninha\\Shapes bacias R00 - teste\\E_BACIA.gpkg'

In [ ]:
# 2) EEB (ponto)
preencher_de_semente(
    caminho_gpkg_entrada = os.path.join(repopath, "EEB.gpkg"),
    layer_entrada        = "EEB",
    caminho_gpkg_semente = os.path.join(repopath, "E_EEB_Semente.gpkg"),
    layer_semente        = "E_EEB_Semente",
    caminho_planilha     = os.path.join(repopath, "EEB.xlsx"),
    chave_gis            = "NOME",
    chave_xls            = "NOME",
    caminho_saida_gpkg   = os.path.join(repopath, "E_EEB.gpkg"),
    layer_saida          = "E_EEB",
)

In [ ]:
# 3) ETE (ponto)
preencher_de_semente(
    os.path.join(repopath, "ETE.gpkg"), "ETE",
    os.path.join(repopath, "E_ETE_Semente.gpkg"), "E_ETE",
    os.path.join(repopath, "ETE.xlsx"),
    chave_gis="NOME", chave_xls="NOME",
    caminho_saida_gpkg=os.path.join(repopath, "E_ETE.gpkg"),
    layer_saida="E_ETE",
)

In [ ]:
# 4) AREAPROJETO (polígono)
preencher_de_semente(
    os.path.join(repopath, "AREAPROJETO.gpkg"), "AREAPROJETO",
    os.path.join(repopath, "E_AREAPROJETO_Semente.gpkg"), "E_AREAPROJETO",
    os.path.join(repopath, "AREAPROJETO.xlsx"),
    chave_gis="NOME", chave_xls="NOME",
    caminho_saida_gpkg=os.path.join(repopath, "E_AREAPROJETO.gpkg"),
    layer_saida="E_AREAPROJETO",
)

In [ ]:
# 5) REDE (linha)
preencher_de_semente(
    os.path.join(repopath, "REDE.gpkg"), "REDE",
    os.path.join(repopath, "E_REDE_Semente.gpkg"), "E_REDE",
    os.path.join(repopath, "REDE.xlsx"),
    chave_gis="NOME", chave_xls="NOME",
    caminho_saida_gpkg=os.path.join(repopath, "E_REDE.gpkg"),
    layer_saida="E_REDE",
)